In [52]:
# ZADANIE 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib.import_data import load_market, MarketEngine

class PricingEngine:
    def __init__(self, market_file: str, tenor: str, imply_pln: bool):
        self.market = load_market(market_file, tenor)
        self.m_engine = MarketEngine(self.market)

        self.r_pln, self.r_eur, self.df_d, self.df_f = (self.m_engine.rates_dfs(imply_pln))

        self.sigma_atm = self.market.atm
        self.sigma_25C = self.market.atm + self.market.bf25 + 0.5 * self.market.rr25
        self.sigma_25P = self.market.atm + self.market.bf25 - 0.5 * self.market.rr25

engine = {
    "3M": PricingEngine("data/market_data.xlsx", tenor="3M", imply_pln=True),
    "6M": PricingEngine("data/market_data.xlsx", tenor="6M", imply_pln=True),
    "9M": PricingEngine("data/market_data.xlsx", tenor="9M", imply_pln=True),
    "1Y": PricingEngine("data/market_data.xlsx", tenor="1Y", imply_pln=True)
}

# RAN

Rozważamy uproszczony wariant Range Accural Note – podstawowy z przepływami nominałów. Wypłata zadana jest wzorem:
$$
N(T_i-T_{i-1})\frac{k_i}{K_i},
$$
gdzie $k_i$ jest liczbą obserwacji, w których kurs $S_t$ znajduje się w korytarzu $L < S_t < L$, a K_i liczbą wszystkich obserwacji.

W ostatnim okresie $T_n$ prócz wypłaty okre´slonej tym wzorem otrzymuje zwrot zainwestowanej kwoty.

Bez utraty ogólności przyjmujemy $N=1$.

Na potrzeby zadania interesuje nas przypadek szczególny:
-  Jeden roczny okres odsetkowy o długości 365 dni. $T_1 = 1$, $T_0 = 0$.
-  $R = 0.08$.
-  $L = S_0 - 0.125$, gdzie $S_0$ to FX Spot (w danych równy 4.3237).
-  Obserwacje $k_i$ mają miejsce w 3M, 6M, 9M, 1Y.
-  U pozostaje parametrem.

## Replikacja

Any zreplikować omawiany instrument wystarczy dla każdego z punktów t = 3M, 6M, 9M, 1Y:
- Zakupić opcję call CoN o momencie zapadnięcia t i K = U
- Sprzedać opcję call CoN o momencie zapadnięcia t i K = L
- Różnica wartości wynikająca z zakupu i sprzedaży jest dyskontowana czynnikiem $D(t, T)$

In [2]:
e = engine["3M"]

L = e.market.spot - 0.125
R = 0.08

## Przypadek z płaskim uśmiechem zmienności

Ponieważ opcja tworzona jest przez replikację, najpierw porównam wartość teoretyczną z wartością replikowaną dla różnych wartości $dK$.

In [13]:
from lib.vanilla_option import VanillaOptionBlackSholes, OptionType
from lib.binary_option import CoNBlackScholes

df_d = e.df_d
df_f = e.df_f
sigma_atm = e.sigma_atm
dK_list = [(1/10)**k for k in range(7)]
T = e.market.expiry
t = e.market.start
S_t = e.market.spot

con_option = CoNBlackScholes(T=T, K=L, option_type=OptionType.CALL)
theoretical_price = con_option.price(
    df_d=df_d,
    df_f=df_f,
    S_t=S_t,
    sigma=sigma_atm,
    t=t,
    base=365
) 
print(f"{'dK':>12}{'Replicated':>18}")
print("-" * 30)
for dK in dK_list:
    option_buy = VanillaOptionBlackSholes(T=T, K=L-dK, option_type=OptionType.CALL)
    option_sell = VanillaOptionBlackSholes(T=T, K=L+dK, option_type=OptionType.CALL)
    buy_price = option_buy.price(
        df_d=df_d,
        df_f=df_f,
        S_t=S_t,
        sigma=sigma_atm,
        t=t,
        base=365
    )
    sell_price = option_sell.price(
        df_d=df_d,
        df_f=df_f,
        S_t=S_t,
        sigma=sigma_atm,
        t=t,
        base=365
    )
    replicated_value = (buy_price - sell_price) / (2*dK)
    print(f"{dK:>12.6f}{replicated_value:>18.10f}")
print(f"{'theoretical':>12}{theoretical_price:>18.10f}")

          dK        Replicated
------------------------------
    1.000000      0.5877319490
    0.100000      0.7714491090
    0.010000      0.7769701843
    0.001000      0.7770263638
    0.000100      0.7770269257
    0.000010      0.7770269313
    0.000001      0.7770269317
 theoretical      0.7770269314


Jak widać replikacja daje bardzo zadawalającą dokładność.

In [51]:
from datetime import date
from typing import Callable

from lib.vanilla_option import VanillaOptionBlackSholes, OptionType

def price_via_replication_flat(
    L: float,
    U: float,
    R: float,
    engine: dict[str, PricingEngine],
    dK: float = 1e-5,
):
    r_per_hit = R / 4.
    df_exp = engine["1Y"].df_d
    value = -1.0 + df_exp
    
    for tenor in ("3M", "6M", "9M", "1Y"):
        e = engine[tenor]
        T = e.market.expiry
        options = [
            VanillaOptionBlackSholes(T=T, K=L-dK, option_type=OptionType.CALL), #b
            VanillaOptionBlackSholes(T=T, K=L+dK, option_type=OptionType.CALL), #s
            VanillaOptionBlackSholes(T=T, K=U+dK, option_type=OptionType.CALL), #b
            VanillaOptionBlackSholes(T=T, K=U-dK, option_type=OptionType.CALL), #s
        ]
        for i, option in enumerate(options):
            price = option.price(
                df_d=e.df_d,
                df_f=e.df_f,
                S_t=e.market.spot,
                sigma=e.sigma_atm,
                t=e.market.start,
                base=365
            )
            value += ((-1) ** i) * r_per_hit * (df_exp / e.df_d) * price / (2*dK)
    return value

def find_U(
    pricing_function: Callable,
    L: float,
    R: float,
    engine: dict[str, PricingEngine],
    dK: float = 1e-5,
    tol: float = 1e-9,
    max_iter: int = 100,
) -> float:
    low = float(L)
    high = float(L * 100.0)
    for _ in range(max_iter):
        mid = 0.5 * (high + low)
        value = pricing_function(L, mid, R, engine)
        if abs(value) < tol:
            return mid
        if value > 0:
            high = mid
        else:
            low = mid
    msg = "max_iter reached without convergence"
    raise ValueError(msg)

U = find_U(price_via_replication_flat, L, R, engine)
print(f"Wartość U wynikająca z optymalizacji: {U:.3f}")

Wartość U wynikająca z optymalizacji: 4.666


## Porównanie U z wynikami symulacji MC

In [70]:
from lib.dates import year_fraction

# funkcja pisana z pomocą AI
def mc_value(
    L: float,
    U: float,
    R: float,
    engine: dict[str, PricingEngine],
    n: int = 10_000,
    seed: int = 42
):
    r_per_hit = R / 4.
    df_exp = engine["1Y"].df_d
    v = np.ones(n, dtype=float) * (-1.0 + df_exp)
    s = np.ones(n, dtype=float) * engine["1Y"].market.spot
    rng = np.random.default_rng(seed)
    prev_tau = 0.0
    df_d_prev, df_f_prev, prev_var = 1.0, 1.0, 0.0
    for i, tenor in enumerate(("3M", "6M", "9M", "1Y")):
        e = engine[tenor]
        z = rng.standard_normal(n)
        tau = year_fraction(e.market.start, e.market.expiry, 365)
        dt = tau - prev_tau
        prev_tau = tau
        sigma2 = e.sigma_atm * e.sigma_atm
        var = sigma2 * tau - prev_var
        df_d=e.df_d
        df_f=e.df_f
        drift = (df_d_prev/df_d) / (df_f_prev/df_f)
        df_d_prev, df_f_prev, prev_var = df_d, df_f, sigma2 * tau
        s = s * drift * np.exp(-0.5*var + np.sqrt(var)*z)
        v += ((s < U) & (s > L)) * r_per_hit * df_exp
    return v.mean()
    
mc_value(L, U, R, engine)

np.float64(5.6582898489667176e-05)

## Cena z replikacji VV

In [ ]:
# def price_via_replication(
#     L: float,
#     U: float,
#     R: float,
#     engine: dict[str, PricingEngine],
#     dK: float = 1e-5,
# ):
#     r_per_hit = R / 4.
#     df_exp = engine["1Y"].df_d
#     value = -1.0 + df_exp
    
#     for tenor in ("3M", "6M", "9M", "1Y"):
#         e = engine[tenor]
#         T = e.market.expiry
#         options = [
#             VanillaOptionBlackSholes(T=T, K=L-dK, option_type=OptionType.CALL), #b
#             VanillaOptionBlackSholes(T=T, K=L+dK, option_type=OptionType.CALL), #s
#             VanillaOptionBlackSholes(T=T, K=U+dK, option_type=OptionType.CALL), #b
#             VanillaOptionBlackSholes(T=T, K=U-dK, option_type=OptionType.CALL), #s
#         ]
#         for i, option in enumerate(options):
#             price = option.price(
#                 df_d=e.df_d,
#                 df_f=e.df_f,
#                 S_t=e.market.spot,
#                 sigma=e.sigma_atm,
#                 t=e.market.start,
#                 base=365
#             )
#             value += ((-1) ** i) * r_per_hit * (df_exp / e.df_d) * price / (2*dK)
#     return value